In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
from pathlib import Path

# =========================
# 1) Paths
# =========================
base = "../../../data/training/"

X_paths = [
    base + "NormalLoad_training_X.npy",
    base + "HighLoad_training_X.npy",
    base + "CriticalLoad_training_X.npy",
]

y_paths = [
    base + "NormalLoad_training_y.npy",
    base + "HighLoad_training_y.npy",
    base + "CriticalLoad_training_y.npy",
]

# =========================
# 2) Load & concat
# =========================
X_list = [np.load(p, allow_pickle=True) for p in X_paths]
y_list = [np.load(p, allow_pickle=True) for p in y_paths]

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

# =========================
# 3) Flatten X if needed
#    (handles N,1,1 → N,1; N,1,4 → N,4; N,4 stays N,4)
# =========================
if X.ndim >= 2:
    X = X.reshape(X.shape[0], -1)

# =========================
# 4) Map labels to 0/1/2
#    Accepts either numeric already or strings with various spellings
# =========================
def to_int_label(lbl):
    if isinstance(lbl, (int, np.integer)):
        return int(lbl)
    s = str(lbl).lower().replace(" ", "").replace("_", "")
    if "normal" in s:
        return 0
    if "high" in s:
        return 1
    if "critical" in s:
        return 2
    raise ValueError(f"Unrecognized label: {lbl}")

y_mapped = np.array([to_int_label(v) for v in y], dtype=int)

# (Optional) sanity print
unique, counts = np.unique(y_mapped, return_counts=True)
print("Class distribution (after mapping):", dict(zip(unique, counts)))

# =========================
# 5) Train RandomForest
# =========================
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf.fit(X, y_mapped)

# =========================
# 6) Evaluate on training set (mesma estrutura do anterior)
#    — se quiseres split/teste real, diz que eu ajusto.
# =========================
y_pred = rf.predict(X)
print("Training Accuracy:", accuracy_score(y_mapped, y_pred))

target_names = ["NormalLoad (0)", "HighLoad (1)", "CriticalLoad (2)"]
print("\nClassification Report:\n",
      classification_report(y_mapped, y_pred, target_names=target_names, digits=4))

# =========================
# 7) Save model
# =========================
save_dir = Path("../../../models/Model3")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "model3_randomforest.pkl"

# guardo também o mapeamento para reutilizar
payload = {"model": rf, "label_map": {"NormalLoad": 0, "HighLoad": 1, "CriticalLoad": 2}}
joblib.dump(payload, save_path)
print(f"Model saved to: {save_path}")


Class distribution (after mapping): {np.int64(0): np.int64(18000), np.int64(1): np.int64(18000), np.int64(2): np.int64(18000)}
Training Accuracy: 1.0

Classification Report:
                   precision    recall  f1-score   support

  NormalLoad (0)     1.0000    1.0000    1.0000     18000
    HighLoad (1)     1.0000    1.0000    1.0000     18000
CriticalLoad (2)     1.0000    1.0000    1.0000     18000

        accuracy                         1.0000     54000
       macro avg     1.0000    1.0000    1.0000     54000
    weighted avg     1.0000    1.0000    1.0000     54000

Model saved to: ..\..\..\models\Model3\model3_randomforest.pkl
